In [1]:
!pip install -q datasets transformers torch evaluate bert_score nltk rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import json
import random
import numpy as np
from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Downloading the ToTTo data from:
https://github.com/google-research-datasets/totto

In [3]:
! wget https://storage.googleapis.com/totto-public/totto_data.zip
!unzip totto_data.zip

--2026-04-15 13:26:46--  https://storage.googleapis.com/totto-public/totto_data.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.202.207, 173.194.203.207, 74.125.199.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.202.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 187724372 (179M) [application/zip]
Saving to: ‘totto_data.zip’

totto_data.zip      100%[===================>] 179.03M   223MB/s    in 0.8s    

2026-04-15 13:26:47 (223 MB/s) - ‘totto_data.zip’ saved [187724372/187724372]

Archive:  totto_data.zip
  inflating: totto_data/totto_dev_data.jsonl  
  inflating: totto_data/totto_train_data.jsonl  
  inflating: totto_data/unlabeled_totto_test_data.jsonl  


In [4]:
!ls totto_data/

totto_dev_data.jsonl  totto_train_data.jsonl  unlabeled_totto_test_data.jsonl


In [5]:
import json

# Read the first 3 lines
with open("totto_data/totto_train_data.jsonl", "r") as f:
    for i, line in enumerate(f):
        exemple = json.loads(line)  # each line -> Python dict
        print(f"\n=== Example {i} ===")
        print(exemple)
        if i >= 2:
            break


=== Example 0 ===
{'table': [[{'value': '#', 'is_header': True, 'column_span': 1, 'row_span': 1}, {'value': 'Run', 'is_header': True, 'column_span': 1, 'row_span': 1}, {'value': 'Title', 'is_header': True, 'column_span': 1, 'row_span': 1}, {'value': 'Chapters', 'is_header': True, 'column_span': 1, 'row_span': 1}, {'value': 'Author', 'is_header': True, 'column_span': 1, 'row_span': 1}, {'value': 'Director', 'is_header': True, 'column_span': 1, 'row_span': 1}, {'value': 'Ibope Rating', 'is_header': True, 'column_span': 1, 'row_span': 1}], [{'value': '59', 'is_header': False, 'column_span': 1, 'row_span': 1}, {'value': 'June 5, 2000— February 2, 2001', 'is_header': False, 'column_span': 1, 'row_span': 1}, {'value': 'Laços de Família', 'is_header': False, 'column_span': 1, 'row_span': 1}, {'value': '209', 'is_header': False, 'column_span': 1, 'row_span': 1}, {'value': 'Manoel Carlos', 'is_header': False, 'column_span': 1, 'row_span': 1}, {'value': 'Ricardo Waddington', 'is_header': False,

In [6]:
import json
with open("totto_data/totto_train_data.jsonl", "r") as f:
    exemple = json.loads(f.readline())  # first line only
    print("Keys :", list(exemple.keys()))

Keys : ['table', 'table_webpage_url', 'table_page_title', 'table_section_title', 'table_section_text', 'highlighted_cells', 'example_id', 'sentence_annotations']


## Table Serialization

In [7]:
import json

def serialize_table(table, highlighted_cells):
    """
    Serializes highlighted cells into linear text.
    Converts a ToTTo table into structured linear text.
    Focuses on highlighted_cells, which are relevant for the description.

    Format: "header1 : value1 | header2 : value2 | ..."
    """
    # Extract headers from row 0
    headers = [cell["value"].strip() for cell in table[0]]

    parts = []
    for (row_idx, col_idx) in highlighted_cells:
        if row_idx >= len(table) or col_idx >= len(table[row_idx]):
            continue
        value = table[row_idx][col_idx]["value"].strip()
        header = headers[col_idx] if col_idx < len(headers) else f"col_{col_idx}"
        if value:
            parts.append(f"{header} : {value}")

    return " | ".join(parts) if parts else "no data"


# Test on the first 3 examples
with open("totto_data/totto_train_data.jsonl", "r") as f:
    for i, line in enumerate(f):
        ex = json.loads(line)
        serialized = serialize_table(ex["table"], ex["highlighted_cells"])
        reference = ex["sentence_annotations"][0]["final_sentence"]
        print(f"\n=== Example {i} ===")
        print(f"TABLE   : {serialized}")
        print(f"REF     : {reference}")
        if i >= 2:
            break


=== Example 0 ===
TABLE   : Title : A Favorita
REF     : A Favorita is the telenovela aired in the 9 pm timeslot.

=== Example 1 ===
TABLE   : Year : 2018 | Player name : Roquan Smith | Position : Linebacker | College : Georgia
REF     : The Chicago Bears recent first round selection (2018) was Roquan Smith, an inside linebacker from Georgia.

=== Example 2 ===
TABLE   : Opponent : Chris Lytle | Event : UFC 127
REF     : Ebersole made his UFC debut against Chris Lytle at UFC 127.


In [8]:
def load_jsonl(path):
    """Loads a jsonl file into a list of dictionaries"""
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data


# Load datasets
train_data = load_jsonl("totto_data/totto_train_data.jsonl")
validation_data = load_jsonl("totto_data/totto_dev_data.jsonl")  # Note: dev = validation

# Dictionary of splits
dataset = {
    "train": train_data,
    "validation": validation_data
}

In [9]:
import random
# Building subsets
def build_subset(split, n):
    """Extracts n valid examples (with at least one highlighted_cell)."""
    subset = []

    for ex in dataset[split]:
        if len(ex["highlighted_cells"]) == 0:
            continue
        serialized = serialize_table(ex["table"], ex["highlighted_cells"])
        reference = ex["sentence_annotations"][0]["final_sentence"]
        if not serialized or not reference:
            continue
        subset.append({
            "table_id": ex.get("table_page_title", ""),
            "serialized_table": serialized,
            "reference": reference,
        })

        if len(subset) >= n:
            break

    return subset

# Build the two subsets
random.seed(42)
train_subset = build_subset("train", 5000)
eval_subset  = build_subset("validation", 500)
print(f"Train subset : {len(train_subset)} examples")
print(f"Eval subset  : {len(eval_subset)} examples")

# Verification
print("\n--- Example eval[0] ---")
print("Table    :", eval_subset[0]["serialized_table"][:120], "...")
print("Ref      :", eval_subset[0]["reference"])

random.shuffle(dataset["train"])
random.shuffle(dataset["validation"])

Train subset : 5000 examples
Eval subset  : 500 examples

--- Example eval[0] ---
Table    : # : 76 | Took Office : Daniel Henry Chamberlain | Left Office : December 1, 1874 ...
Ref      : Daniel Henry Chamberlain was the 76th Governor of South Carolina from 1874.


## Local Saving in Colab

In [10]:
import os

#os.makedirs("/content/drive/MyDrive/genai", exist_ok=True)

In [11]:
with open("/content/drive/MyDrive/genai/train_subset.json", "w") as f:
    json.dump(train_subset, f, ensure_ascii=False)

with open("/content/drive/MyDrive/genai/eval_subset.json", "w") as f:
    json.dump(eval_subset, f, ensure_ascii=False)

print("Subsets saved to Google Drive (genai folder).")

Subsets saved to Google Drive (genai folder).


In [12]:
!ls /content/drive/MyDrive/genai

eval_subset.json  train_subset.json
